# 五个数据集 daft 加载总览

选择算子课题 `f=φ∘a` 用到的全部本地 parquet,用 daft 统一 load 并展示。

| 数据集 | 角色 | 路径 |
|---|---|---|
| CR9114 | 亲和力 a (H1/H3/B) | `cr9114/datasets/cr9114/` |
| CR6261 | 亲和力 a (H1/H9) | `cr9114/datasets/cr6261/` |
| MAGMA-seq | 亲和力 a (多抗体×多抗原) | `magma-seq/datasets/magma_seq/` |
| Engelhart | 亲和力 a (scFv×SARS) | `engelhart/datasets/engelhart/` |
| S5F | 突变算子 M (5-mer SHM) | `s5f/datasets/s5f/` |

各数据集保持原生 schema,本轮未做跨集统一。

In [1]:
import daft
from pathlib import Path

# context/ 在 OUROBOROS-AI4S 根下, 各数据集是其 sibling
ROOT = Path.cwd()
if ROOT.name == "context":
    ROOT = ROOT.parent
if not (ROOT / "cr9114").exists():
    ROOT = Path("g:/OUROBOROS-AI4S")

PATHS = {
    "cr9114":    ROOT / "cr9114"   / "datasets" / "cr9114",
    "cr6261":    ROOT / "cr9114"   / "datasets" / "cr6261",
    "magma_seq": ROOT / "magma-seq" / "datasets" / "magma_seq",
    "engelhart": ROOT / "engelhart" / "datasets" / "engelhart",
    "s5f":       ROOT / "s5f"      / "datasets" / "s5f",
}
for k, p in PATHS.items():
    print(f"{k:12s} {'OK ' if p.exists() else 'MISSING'} {p}")

cr9114       OK  G:\OUROBOROS-AI4S\cr9114\datasets\cr9114
cr6261       OK  G:\OUROBOROS-AI4S\cr9114\datasets\cr6261
magma_seq    OK  G:\OUROBOROS-AI4S\magma-seq\datasets\magma_seq
engelhart    OK  G:\OUROBOROS-AI4S\engelhart\datasets\engelhart
s5f          OK  G:\OUROBOROS-AI4S\s5f\datasets\s5f


In [2]:
# daft 读各 parquet 目录/文件 (目录与单文件 daft 均可直接 read_parquet)
def load(p: Path) -> daft.DataFrame:
    target = str(p) if p.is_file() else str(p / "**/*.parquet")
    return daft.read_parquet(target)

dfs = {k: load(p) for k, p in PATHS.items()}
{k: df.count_rows() for k, df in dfs.items()}

[00:00] 

[00:00] 

[00:00] 

[00:00] 

[00:00] 

{'cr9114': 65536,
 'cr6261': 2048,
 'magma_seq': 3194,
 'engelhart': 1259700,
 's5f': 1024}

In [3]:
# 每个数据集: schema + 行数 + 前几行
for k, df in dfs.items():
    print("=" * 70)
    print(f"### {k}   rows={df.count_rows()}")
    print("-" * 70)
    for f in df.schema():
        print(f"  {f.name:14s} {f.dtype}")
    print()

[00:00] 


### cr9114   rows=65536
----------------------------------------------------------------------
  genotype       String
  som_mut        Int8
  h1_mean        Float64
  h1_sem         Float64
  h1_censored    Bool
  h3_mean        Float64
  h3_sem         Float64
  h3_censored    Bool
  b_mean         Float64
  b_sem          Float64
  b_censored     Bool



[00:00] 


### cr6261   rows=2048
----------------------------------------------------------------------
  genotype       String
  som_mut        Int8
  h1_mean        Float64
  h1_sem         Float64
  h1_censored    Bool
  h9_mean        Float64
  h9_sem         Float64
  h9_censored    Bool



[00:00] 


### magma_seq   rows=3194
----------------------------------------------------------------------
  antibody       String
  variant        String
  antigen        String
  Kd_nM          Float64
  Fmax           Float64
  ddg            Float64
  ci_low         Float64
  ci_high        Float64
  n              Int64
  avg_counts     Float64
  success        Bool
  source_sheet   String



[00:00] 


### engelhart   rows=1259700
----------------------------------------------------------------------
  POI            String
  Sequence       String
  HC             String
  LC             String
  CDRH1          String
  CDRH2          String
  CDRH3          String
  CDRL1          String
  CDRL2          String
  CDRL3          String
  Target         String
  Assay          Int64
  Replicate      Int64
  Pred_affinity  Float64



[00:00] 


### s5f   rows=1024
----------------------------------------------------------------------
  fivemer        String
  mutability     Float64
  sub_A          Float64
  sub_C          Float64
  sub_G          Float64
  sub_T          Float64



## CR9114 — 亲和力 a (16 位点库, H1/H3/B, −logKD 越高越强)

In [4]:
dfs["cr9114"].show(8)

genotypeString,som_mutInt8,h1_meanFloat64,h1_semFloat64,h1_censoredBool,h3_meanFloat64,h3_semFloat64,h3_censoredBool,b_meanFloat64,b_semFloat64,b_censoredBool
1001110010000101,7,9.484585780732429,0.011475670303133671,false,6,0,true,6,0,true
0001111110011011,10,9.430990995957574,0.042676502386221685,false,6,0,true,6,0,true
1111110010011111,12,9.497006442037117,0.030798001461830187,false,6.736212784175028,0.01924590866477363,false,6,0,true
1011111110101110,12,9.499691425717383,0.02885219011062568,false,6,0,true,6,0,true
1111101010000111,10,9.380737338620486,0.011946943573310994,false,6,0,true,6,0,true
1111010011110011,11,9.52902022022239,0.01957566051394931,false,7.892638721552122,0.018606446023465957,false,6,0,true
1100101011111101,11,9.58893633625694,0.010400760655518928,false,6,0,true,6,0,true
1011010010101001,8,9.39740406571441,0.009023271875365774,false,6,0,true,6,0,true


## CR6261 — 亲和力 a (11 位点库补齐满库 2048, H1/H9; 缺基因型行为 null)

In [5]:
dfs["cr6261"].show(8)

genotypeString,som_mutInt8,h1_meanFloat64,h1_semFloat64,h1_censoredBool,h9_meanFloat64,h9_semFloat64,h9_censoredBool
00000000000,0,7,0,true,7,0,true
00000000001,1,7,0,true,7,0,true
00000000010,1,7,0,true,7,0,true
00000000011,2,7,0,true,7,0,true
00000000100,1,7,0,true,7,0,true
00000000101,2,7,0,true,7,0,true
00000000110,2,8.549745180506626,0.08321914711428392,false,7.068712925208931,0.03967142586618314,false
00000000111,3,8.543072711756809,0.06571090124166534,false,7.441028811858243,0,false


## MAGMA-seq — 亲和力 a (9 抗体 × S1/HA/N2; Kd 原生 nM 越低越强)

In [6]:
dfs["magma_seq"].show(8)
# 抗体/抗原分布
dfs["magma_seq"].groupby("antibody", "antigen").count("variant").show(20)

antibodyString,antigenString,variantUInt64
4A8,S1,74
CC121,S1,83
Ab_2-17,HA,353
1G04,N2,107
Ab_2-7,HA,1
CR6261,HA,388
319-345,HA,1319
222-1C06,HA,808
1G01,N2,61


## Engelhart — 亲和力 a (scFv × SARS; Pred_affinity=log10 Kd nM 越低越强, ~1.26M 行多 NaN)

In [7]:
dfs["engelhart"].select("POI", "Target", "Assay", "Replicate", "Pred_affinity", "CDRH3").show(8)
dfs["engelhart"].groupby("Target").count("POI").show()

TargetString,POIUInt64
AlphaNeg3,314925
MIT_Target,314925
AlphaNeg2,314925
AlphaNeg1,314925


## S5F — 突变算子 M (1024 个 5-mer; mutability + 替换四向量, 中心碱基自身=null)

In [8]:
dfs["s5f"].show(10)

fivemerString,mutabilityFloat64,sub_AFloat64,sub_CFloat64,sub_GFloat64,sub_TFloat64
AAAAA,0.0018210107848663,None,0.3543814433,0.4149484536,0.2306701031
AAAAC,0.0011448782843769,None,0.2378854626,0.4757709251,0.2863436123
AAAAG,0.001430707487307,None,0.2419700214,0.4582441113,0.2997858672
AAAAT,0.0031201520737356,None,0.6119966443,0.2432885906,0.1447147651
AAACA,0.0011219412361346,None,0.25,0.4810606061,0.2689393939
AAACC,0.0011740762734394,None,0.1867424242,0.4823863636,0.3308712121
AAACG,0.0021289426402379,None,0.2783825816,0.4987039917,0.2229134266
AAACT,0.0006434804649424,None,0.2627118644,0.4830508475,0.2542372881
AAAGA,0.001147328026369,None,0.203125,0.5204326923,0.2764423077
AAAGC,0.0014794143247094,None,0.1770920211,0.4840144565,0.3388935224


---
# 目标:`f = φ∘a` 任务需要的统一 dataset schema

上面 5 个数据集各自原生落盘了。现在说明:**要训练整条 `f=φ∘a`,我们最终需要把它们 ETL 成什么形状。**

## 先看主方程,字段需求从它推出来

抗体在亲和力成熟中的演化,是一个**突变-选择主方程**(quasispecies / replicator-mutator):

```
rate(s → s')  =  M(s → s')  ×  exp[ f(s', c) − f(s, c) ]
                 └ 已知突变算子   └ 待学的条件依赖选择势,f = φ∘a
```

- **a(序列, 条件)** = 亲和力,**住在节点上**。一个 `(变体 s, 条件 c)` 配一个亲和力测量。
- **f = φ∘a** = 选择势,也**住在节点上**:由该节点亲和力经单调映射 φ 得到。
- **M(s→s')** = SHM 突变率,**住在边上**:`s→s'` 是一次单点突变,由突变位点的**核苷酸 5-mer 上下文**决定(= S5F)。

所以统一 dataset 不是"把 5 张表拼宽",而是**一张带条件的序列图**:三个亲和力数据集填**节点**,S5F 填**边**。下面的 schema 就是这个图的两类表 + 一张词表。

## 三层字段需求(对照上面 5 个真实 schema)

| 层 | 需要的字段 | 当前哪些数据集已提供 | 缺口 |
|---|---|---|---|
| **节点身份** | 抗体锚点、序列(AA + 尽量核苷酸)、突变列表 | 全部有 AA;**仅 MAGMA 有密码子** | CR9114/Engelhart 缺核苷酸 → 边上 M 难算 |
| **节点测量 a** | 亲和力(统一标度)、误差、删失方向、条件 c=抗原 | 全部有亲和力 | **标度/符号不统一**(见下) |
| **边 + M** | `s→s'` 单点突变、5-mer 上下文、mutability、替换概率 | S5F 提供 M 查找表 | 边本身待物化(组合的) |

**标度不统一是核心痛点**(本轮特意没动,留给统一层):
- CR9114: `−logKD`,**越高越强**
- MAGMA: `Kd_nM` 绝对值,**越低越强**
- Engelhart: `log10(Kd nM)`,**越低越强**

统一层须归一到单一标度(建议 `pKd = −log10(Kd[M])`,越高越强),并把各自的检测限/饱和统一成**删失方向**字段。

下面 cell 是这个统一 schema 的 Python 伪代码(dataclass 形式,仅说明字段,不是可运行 ETL)。


In [ ]:
# ============================================================================
#  f = φ∘a 任务的统一 dataset schema —— 伪代码(仅说明字段,不可运行)
#  图实现: 节点表(亲和力 a) + 边表(突变 M) + 词表(抗原/抗体规范化)
# ============================================================================
from __future__ import annotations
from dataclasses import dataclass
from enum import Enum


class CensorDir(Enum):
    """删失方向 —— 统一三个数据集各自的检测限/饱和语义。"""
    NONE  = "none"    # 精确测量
    LEFT  = "left"    # 弱结合, 低于检测下限(真值落在 [-inf, floor])
    RIGHT = "right"   # 强结合, 高于饱和上限(真值落在 [ceil, +inf])


# ---------------------------------------------------------------------------
# 节点表: 一行 = 一个 (变体 s, 条件 c=抗原) 的亲和力测量。喂 a / f。
#   CR9114 / CR6261 / MAGMA / Engelhart 全部 unpivot 进这张表。
# ---------------------------------------------------------------------------
@dataclass
class AffinityNode:
    # —— 身份: 把变体唯一钉死在某抗体的突变坐标系里 ——
    node_id:      str          # 稳定主键 = hash(library, mutations, antigen)
    library:      str          # 母本抗体锚点: "CR9114"/"CR6261"/"4A8"/"1G01"... (突变坐标原点)
    mutations:    list[str]    # 规范化突变列表, 每条 "链:WT位点MUT" 如 ["VH:M59I","VL:T94M"]; 母本=[]
    n_mut:        int          # = len(mutations)
    aa_vh:        str          # VH 氨基酸全长序列
    aa_vl:        str | None   # VL 全长; 单域(CR9114/61 仅重链, VHH)为 None
    nt_vh:        str | None   # VH 核苷酸序列(算边上 M 的必需品); CR9114/Engelhart 暂缺→None
    nt_vl:        str | None   # VL 核苷酸; 多数缺→None

    # —— 条件 c ——
    antigen:      str          # 规范化抗原名(do-干预/条件轴): "H1"/"H3"/"B"/"H9"/"S1"/"HA"/"N2"...

    # —— 测量 a(统一标度) ——
    affinity:     float | None # pKd = −log10(Kd[M]); 越高=越强(全数据集归一后); 未测=None
    affinity_sem: float | None # 测量误差(有则填; CR9114 Tite-Seq SE≈0.047)
    censored:     bool         # 是否删失
    censor_dir:   CensorDir    # 删失方向(见上)

    # —— 溯源(PROV: 每行自解释、可复现) ——
    source:       str          # 整串来源描述, 如 "magma_seq/topbinHA_rep2" / "cr9114/fig1-data1-v3"
    assay:        str          # 测量平台: "Tite-Seq"/"MAGMA-seq"/"AlphaSeq"
    quality:      float | None # 质量/置信(MAGMA success、Engelhart replicate 一致性等, 归一到[0,1])


# ---------------------------------------------------------------------------
# 边表: 一行 = 一次单点突变 s→s'。承载已知算子 M。喂主方程似然。
#   边由节点表在同一 library 内按 Hamming-1 邻接生成; M 由 S5F 按 5-mer 上下文查得。
# ---------------------------------------------------------------------------
@dataclass
class MutationEdge:
    src_node_id:  str          # 起点变体 (→ AffinityNode.node_id)
    dst_node_id:  str          # 终点变体, 与 src 恰好差一个突变
    library:      str          # 同 src/dst 的母本锚点
    edge_mut:     str          # 这一步突变 "链:WT位点MUT"
    fivemer:      str          # 突变位点的核苷酸 5-mer 上下文(查 S5F 的键); 缺核苷酸时 None
    M_mutability: float        # S5F: 该 5-mer 中心碱基被 SHM 靶向的概率
    M_sub_prob:   float        # S5F: 中心碱基→该终点碱基的替换概率
    # 训练时 Δf 由两端节点的 f=φ(affinity) 之差给出, 不在边表物化(避免重复存)


# ---------------------------------------------------------------------------
# 词表: 解决"同概念不同字符串"(SSSOM 思路)。规范化 + 可审计映射。
# ---------------------------------------------------------------------------
@dataclass
class VocabMap:
    axis:           str        # "antigen" / "library" / "assay"
    source_string:  str        # 原始串, 如 "CC121" / "fluB" / "MIT_Target"
    source_dataset: str        # 来自哪个数据集
    canonical:      str        # 规范术语, 如 "CC12.1" / "B" / "SARS-CoV-2_S1"


# ============================================================================
#  落地形态(下一轮 ETL 的目标):
#    nodes.parquet  ← 4 个亲和力数据集 unpivot + pKd 归一 + 删失方向 + 规范化
#    edges.parquet  ← 在 nodes 上按 library 内 Hamming-1 生成 + S5F 查 M
#    vocab.parquet  ← 抗原/抗体/平台的规范化映射(本轮已隐含在各 schema.py 常量里)
#  三表外键耦合 = 带条件的序列图 = 能同时训练 a / φ / 主方程的统一 dataset。
#
#  仍需补的缺口(本轮已暴露):
#    1) CR9114/Engelhart 缺核苷酸序列 → 边上 M 暂只能在 MAGMA 子图上算
#    2) 边是组合的 → 须决定物化哪些(完备库全物化 / 其余按需采样)
#    3) φ(affinity→fitness) 的函数形式 → 是学习目标, 不在 schema 里固定
# ============================================================================
